# Team Season - team_dashboard_by_general_splits

This notebook inspects the data **downloaded** from the `team_dashboard_by_general_splits` endpoint.

Source folder: `02_data/01_raw/2025_26/02_team_season/team_dashboard_by_general_splits`

Scope:
- Read parquet files in the folder
- Show how many files were downloaded and how many rows/columns they contain
- Report missing values (null cells) overall and by row/column distribution

Notes:
- Read-only analysis: no exports or writes


In [9]:
from pathlib import Path
import pandas as pd
import numpy as np

# Resolve project root by walking up to find 02_data
project_root = Path.cwd()
for _ in range(6):
    if (project_root / "02_data").exists():
        break
    project_root = project_root.parent

endpoint = "team_dashboard_by_general_splits"
data_dir = project_root / "02_data" / "01_raw" / "2025_26" / "02_team_season" / endpoint

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)


In [10]:
files = sorted(data_dir.glob("*.parquet"))
print(f"Endpoint: {endpoint}")
print(f"Folder: {data_dir}")
print(f"Parquet files: {len(files)}")

files_df = pd.DataFrame({
    "file": [f.name for f in files],
    "size_mb": [round(f.stat().st_size / 1e6, 3) for f in files],
})
files_df


Endpoint: team_dashboard_by_general_splits
Folder: /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/01_raw/2025_26/02_team_season/team_dashboard_by_general_splits
Parquet files: 30


,file,size_mb
0,team_dashboard_by_general_splits__team_id=1610...,0.040
1,team_dashboard_by_general_splits__team_id=1610...,0.040
2,team_dashboard_by_general_splits__team_id=1610...,0.040
3,team_dashboard_by_general_splits__team_id=1610...,0.040
4,team_dashboard_by_general_splits__team_id=1610...,0.041
5,team_dashboard_by_general_splits__team_id=1610...,0.041
6,team_dashboard_by_general_splits__team_id=1610...,0.040
7,team_dashboard_by_general_splits__team_id=1610...,0.041
8,team_dashboard_by_general_splits__team_id=1610...,0.041
9,team_dashboard_by_general_splits__team_id=1610...,0.040


In [11]:
dfs = [pd.read_parquet(f) for f in files]

per_file = pd.DataFrame({
    "file": [f.name for f in files],
    "rows": [len(df) for df in dfs],
    "cols": [df.shape[1] for df in dfs],
})
per_file


,file,rows,cols
0,team_dashboard_by_general_splits__team_id=1610...,18,65
1,team_dashboard_by_general_splits__team_id=1610...,17,65
2,team_dashboard_by_general_splits__team_id=1610...,18,65
3,team_dashboard_by_general_splits__team_id=1610...,18,65
4,team_dashboard_by_general_splits__team_id=1610...,19,65
5,team_dashboard_by_general_splits__team_id=1610...,19,65
6,team_dashboard_by_general_splits__team_id=1610...,18,65
7,team_dashboard_by_general_splits__team_id=1610...,18,65
8,team_dashboard_by_general_splits__team_id=1610...,18,65
9,team_dashboard_by_general_splits__team_id=1610...,18,65


In [12]:
df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

print(f"Combined shape: {df.shape}")
print(f"Total rows: {len(df)}")
print(f"Total columns: {df.shape[1] if len(df.columns) else 0}")


Combined shape: (549, 65)
Total rows: 549
Total columns: 65


In [13]:
rows, cols = df.shape
total_cells = rows * cols
null_cells = int(df.isna().sum().sum()) if total_cells else 0
null_pct = (null_cells / total_cells) if total_cells else 0

summary = pd.DataFrame({
    "rows": [rows],
    "cols": [cols],
    "total_cells": [total_cells],
    "null_cells": [null_cells],
    "null_pct": [round(null_pct * 100, 3)],
})
summary


,rows,cols,total_cells,null_cells,null_pct
0,549,65,35685,2745,7.692


In [14]:
if df.empty:
    col_summary = pd.DataFrame()
else:
    col_nulls = df.isna().sum()
    col_null_pct = df.isna().mean()
    col_summary = (
        pd.DataFrame({
            "null_cells": col_nulls,
            "null_pct": (col_null_pct * 100).round(3),
        })
        .sort_values("null_pct", ascending=False)
    )

col_summary


,null_cells,null_pct
SEASON_YEAR,519,94.536
GAME_RESULT,489,89.071
SEASON_SEGMENT,489,89.071
TEAM_GAME_LOCATION,482,87.796
SEASON_MONTH_NAME,399,72.678
TEAM_DAYS_REST_RANGE,367,66.849
FG3_PCT_RANK,0,0.000
DREB_RANK,0,0.000
OREB_RANK,0,0.000
FT_PCT_RANK,0,0.000


In [15]:
if df.empty:
    row_dist = pd.DataFrame()
else:
    row_null_pct = df.isna().mean(axis=1)
    bins = [0, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 1.0]
    row_bins = pd.cut(row_null_pct, bins=bins, include_lowest=True)
    row_counts = row_bins.value_counts().sort_index()
    row_dist = pd.DataFrame({
        "rows": row_counts,
        "pct_rows": (row_counts / len(row_null_pct) * 100).round(3),
    })

row_dist


,rows,pct_rows
"(-0.001, 0.01]",0,0.0
"(0.01, 0.05]",0,0.0
"(0.05, 0.1]",549,100.0
"(0.1, 0.25]",0,0.0
"(0.25, 0.5]",0,0.0
"(0.5, 0.75]",0,0.0
"(0.75, 1.0]",0,0.0


In [16]:
# Merge all teams, fill SEASON, split by TABLE_NAME, drop fully-null columns, report nulls, and export

files = sorted(data_dir.glob("*.parquet"))
dfs = [pd.read_parquet(f) for f in files]
df_all = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

if df_all.empty:
    print("No data to merge.")
else:
    # Fill SEASON using non-null value (typically from Overall rows)
    if "SEASON" in df_all.columns:
        season_value = df_all["SEASON"].dropna().unique()
        season_value = season_value[0] if len(season_value) else None
        if season_value is not None:
            df_all["SEASON"] = df_all["SEASON"].fillna(season_value)
            print(f"Filled SEASON with: {season_value}")
        else:
            print("SEASON is fully null; nothing to fill.")
    else:
        print("SEASON column not found.")

    table_col = "TABLE_NAME" if "TABLE_NAME" in df_all.columns else None
    if table_col is None:
        print("Required column not found: TABLE_NAME")
        print(df_all.columns)
    else:
        out_dir = project_root / "02_data" / "02_cleaned" / "2025_26" / "02_team_season"
        out_dir.mkdir(parents=True, exist_ok=True)

        table_values = sorted([v for v in df_all[table_col].dropna().unique()])

        null_summary = []
        for name in table_values:
            df_part = df_all.loc[df_all[table_col] == name].copy()

            if df_part.empty:
                print(f"\n{name}: no rows")
                fully_null = []
            else:
                col_nulls = df_part.isna().sum()
                col_null_pct = (df_part.isna().mean() * 100).round(3)
                cols_with_nulls = col_nulls[col_nulls > 0].index.tolist()
                fully_null = col_nulls[col_nulls == len(df_part)].index.tolist()

                print(f"\n{name}: columns with any nulls ({len(cols_with_nulls)})")
                if cols_with_nulls:
                    display(pd.DataFrame({
                        "null_cells": col_nulls[cols_with_nulls],
                        "null_pct": col_null_pct[cols_with_nulls],
                    }).sort_values("null_pct", ascending=False))

                print(f"{name}: fully null columns ({len(fully_null)})")
                if fully_null:
                    print(fully_null)

            # Drop fully-null columns (they belong to other tables)
            if fully_null:
                df_part = df_part.drop(columns=fully_null)

            # Recompute null summary AFTER dropping fully-null columns
            rows, cols = df_part.shape
            total_cells = rows * cols
            null_cells = int(df_part.isna().sum().sum()) if total_cells else 0
            null_pct = (null_cells / total_cells * 100) if total_cells else 0

            null_summary.append({
                "table_name": name,
                "rows": rows,
                "cols": cols,
                "total_cells": total_cells,
                "null_cells": null_cells,
                "null_pct": round(null_pct, 3),
            })

            out_path = out_dir / f"team_dashboard_by_general_splits__{name}.parquet"
            df_part.to_parquet(out_path, index=False)
            print(f"Saved {name}: {len(df_part)} rows -> {out_path}")

        null_summary_df = pd.DataFrame(null_summary).sort_values("table_name")
        print("\nNull summary by TABLE_NAME (after dropping fully-null columns):")
        print(null_summary_df)


Filled SEASON with: 2025-26

DaysRestTeamDashboard: columns with any nulls (5)


,null_cells,null_pct
SEASON_YEAR,182,100.0
TEAM_GAME_LOCATION,182,100.0
GAME_RESULT,182,100.0
SEASON_MONTH_NAME,182,100.0
SEASON_SEGMENT,182,100.0


DaysRestTeamDashboard: fully null columns (5)
['SEASON_YEAR', 'TEAM_GAME_LOCATION', 'GAME_RESULT', 'SEASON_MONTH_NAME', 'SEASON_SEGMENT']
Saved DaysRestTeamDashboard: 182 rows -> /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/02_cleaned/2025_26/02_team_season/team_dashboard_by_general_splits__DaysRestTeamDashboard.parquet

LocationTeamDashboard: columns with any nulls (5)


,null_cells,null_pct
SEASON_YEAR,67,100.0
GAME_RESULT,67,100.0
SEASON_MONTH_NAME,67,100.0
SEASON_SEGMENT,67,100.0
TEAM_DAYS_REST_RANGE,67,100.0


LocationTeamDashboard: fully null columns (5)
['SEASON_YEAR', 'GAME_RESULT', 'SEASON_MONTH_NAME', 'SEASON_SEGMENT', 'TEAM_DAYS_REST_RANGE']
Saved LocationTeamDashboard: 67 rows -> /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/02_cleaned/2025_26/02_team_season/team_dashboard_by_general_splits__LocationTeamDashboard.parquet

MonthTeamDashboard: columns with any nulls (5)


...

OverallTeamDashboard: fully null columns (5)
['TEAM_GAME_LOCATION', 'GAME_RESULT', 'SEASON_MONTH_NAME', 'SEASON_SEGMENT', 'TEAM_DAYS_REST_RANGE']
Saved OverallTeamDashboard: 30 rows -> /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/02_cleaned/2025_26/02_team_season/team_dashboard_by_general_splits__OverallTeamDashboard.parquet

PrePostAllStarTeamDashboard: columns with any nulls (5)


,null_cells,null_pct
SEASON_YEAR,60,100.0
TEAM_GAME_LOCATION,60,100.0
GAME_RESULT,60,100.0
SEASON_MONTH_NAME,60,100.0
TEAM_DAYS_REST_RANGE,60,100.0


PrePostAllStarTeamDashboard: fully null columns (5)
['SEASON_YEAR', 'TEAM_GAME_LOCATION', 'GAME_RESULT', 'SEASON_MONTH_NAME', 'TEAM_DAYS_REST_RANGE']
Saved PrePostAllStarTeamDashboard: 60 rows -> /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/02_cleaned/2025_26/02_team_season/team_dashboard_by_general_splits__PrePostAllStarTeamDashboard.parquet

WinsLossesTeamDashboard: columns with any nulls (5)


,null_cells,null_pct
SEASON_YEAR,60,100.0
TEAM_GAME_LOCATION,60,100.0
SEASON_MONTH_NAME,60,100.0
SEASON_SEGMENT,60,100.0
TEAM_DAYS_REST_RANGE,60,100.0


WinsLossesTeamDashboard: fully null columns (5)
['SEASON_YEAR', 'TEAM_GAME_LOCATION', 'SEASON_MONTH_NAME', 'SEASON_SEGMENT', 'TEAM_DAYS_REST_RANGE']
Saved WinsLossesTeamDashboard: 60 rows -> /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/02_cleaned/2025_26/02_team_season/team_dashboard_by_general_splits__WinsLossesTeamDashboard.parquet

Null summary by TABLE_NAME (after dropping fully-null columns):
                    table_name  rows  cols  total_cells  null_cells  null_pct
0        DaysRestTeamDashboard   182    60        10920           0       0.0
1        LocationTeamDashboard    67    60         4020           0       0.0
2           MonthTeamDashboard   150    60         9000           0       0.0
3         OverallTeamDashboard    30    60         1800           0       0.0
4  PrePostAllStarTeamDashboard    60    60         3600           0       0.0
5      WinsLossesTeamDashboard    60    60         3600           0       0.0
